# 04. LSTM raw-waveform music generation

This notebook cleans the most coherent surviving generation experiment, which trained on five stereo recordings from instrument class 1.

Recovered method:

- first 600,000 samples of each WAV loaded;
- first 400,000 samples per recording used for training;
- look-back window of 10 waveform samples;
- separate LSTM models for left and right channels;
- LSTM(100) followed by Dense 50 → 25 → 12 → 1 with LeakyReLU;
- Adam + mean squared error;
- 20 epochs, batch size 100;
- later portions used as seed/evaluation material.

### Correction

The original combined notebook trained `rnn1` and `rnn2` but accidentally used `rnn1` to predict **both** stereo channels. This version uses the correct model for each channel.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


In [ ]:
import numpy as np
from src.generation import load_stereo_wav, make_sequences, build_waveform_lstm, write_stereo_wav

SOURCE_DIR = ROOT / "data" / "generation_sources"
OUTPUT_WAV = ROOT / "data" / "generated" / "pred_rnn2.wav"

SOURCE_FILES = [SOURCE_DIR / f"1.{i}.wav" for i in range(1, 6)]
LOOK_BACK = 10
TRAIN_SAMPLES = 400_000
TOTAL_SAMPLES = 600_000
EPOCHS = 20
BATCH_SIZE = 100


In [ ]:
missing = [p for p in SOURCE_FILES if not p.exists()]
if missing:
    raise FileNotFoundError(
        "The original generation source files were not recovered. Missing:\n"
        + "\n".join(str(p) for p in missing)
    )

loaded = [load_stereo_wav(p) for p in SOURCE_FILES]
sample_rates = {sr for sr, _ in loaded}
if len(sample_rates) != 1:
    raise ValueError(f"Source files have different sample rates: {sample_rates}")
sr = loaded[0][0]
audio = [data[:TOTAL_SAMPLES] for _, data in loaded]


In [ ]:
# Match the recovered combined notebook: concatenate the first 400k
# samples from each recording, then construct next-sample windows.
train = np.concatenate([a[:TRAIN_SAMPLES] for a in audio], axis=0)

X_left, y_left = make_sequences(train[:, 0], LOOK_BACK)
X_right, y_right = make_sequences(train[:, 1], LOOK_BACK)

print(X_left.shape, X_right.shape)


In [ ]:
left_model = build_waveform_lstm(LOOK_BACK)
right_model = build_waveform_lstm(LOOK_BACK)

left_history = left_model.fit(
    X_left, y_left, epochs=EPOCHS, batch_size=BATCH_SIZE
)
right_history = right_model.fit(
    X_right, y_right, epochs=EPOCHS, batch_size=BATCH_SIZE
)


In [ ]:
test = np.concatenate(
    [a[TRAIN_SAMPLES:TOTAL_SAMPLES] for a in audio],
    axis=0,
)

X_left_test, _ = make_sequences(test[:, 0], LOOK_BACK)
X_right_test, _ = make_sequences(test[:, 1], LOOK_BACK)

pred_left = left_model.predict(X_left_test).reshape(-1)
pred_right = right_model.predict(X_right_test).reshape(-1)  # corrected: right model

write_stereo_wav(OUTPUT_WAV, sr, pred_left, pred_right)
print(f"Wrote: {OUTPUT_WAV}")


This is a **raw-waveform next-sample prediction** model, not a symbolic MIDI-note generator. A separate surviving notebook later experimented with forcing outputs toward a pentatonic scale, but that code operates on raw PCM amplitudes while treating them like MIDI note values, so it is preserved only as a legacy experiment rather than presented as a core method.